In [24]:
import numpy as np
import matplotlib.pyplot as plt
import pymcel as pc
import rebound as rb
import plotly.graph_objects as go

### <span style="color:red">Clase 6: Sistema de Unidades Canónicas para Integración N-Cuerpos</span>

Para garantizar la estabilidad numérica durante la integración de las ecuaciones de movimiento, es imperativo realizar una adimensionalización del sistema. En mecánica celeste, esto se logra definiendo un sistema de unidades canónicas donde la constante de gravitación universal es $G = 1$.

Definimos nuestras unidades fundamentales de masa ($MU$) y distancia ($DU$) basadas en el sistema heliocéntrico:
* **Unidad de Masa (MU):** $1 \, M_{\odot}$ (Masa Solar)
* **Unidad de Distancia (DU):** $1 \, \text{AU}$ (Unidad Astronómica)

Para que la ecuación de movimiento $\ddot{\mathbf{r}} = -G \frac{M}{r^3} \mathbf{r}$ retenga su forma sin el factor $G$, la unidad de tiempo ($TU$) queda rígidamente determinada por el análisis dimensional:

$$TU = \sqrt{\frac{DU^3}{G \cdot MU}}$$

Sustituyendo los valores en el Sistema Internacional (SI):

$$TU = \sqrt{\frac{(1.495978707 \times 10^{11} \, \text{m})^3}{(6.67430 \times 10^{-11} \, \text{m}^3 \text{kg}^{-1} \text{s}^{-2}) (1.98847 \times 10^{30} \, \text{kg})}} \approx 5.022 \times 10^6 \, \text{s}$$

Esto equivale aproximadamente a **58.132 días**. Esta unidad canónica de tiempo es el factor de escala de la simulación, mientras que las épocas de los eventos astronómicos se seguirán registrando en **Tiempo Juliano (JD)** para referenciar las fechas calendario reales del acercamiento de Apophis.

Por lo tanto, la unidad de velocidad canónica ($VU$) se define como:

$$VU = \frac{DU}{TU} \approx 29.78 \, \text{km/s}$$

Al usar este sistema, la Tierra viaja aproximadamente a $1 \, VU$ y la distancia Tierra-Sol es $1 \, DU$, manteniendo todos los valores de las matrices de estado en órdenes de magnitud cercanos a la unidad y evitando inestabilidades numéricas.

In [48]:
# Entonces, el sistema de unidades:

G_canon = 1 # Gravitational constant
M_canon = 1 # Mass Unit
D_canon = 1 # Distance unit

G = pc.constantes.G
M_sun = pc.constantes.M_sun
AU = pc.constantes.au

T_canon = np.sqrt(AU**3/(G*M_sun)) # Time unit
V_canon = AU/T_canon # Velocity unit

### Clase 6

Se debe de trabajar con unidades canónicas, y jugar un poquito con ellas !!

 - `tabla_*` keeps the original Horizons units (often AU and AU/day).
 - `pos_vel_*` is converted by pymcel to SI by default: meters and meters/second.
 Convention: state = [x, y, z, vx, vy, vz] (barycentric because location='@0')

Mass from https://web.archive.org/web/20130512035601/http://neo.jpl.nasa.gov/risk/a99942.html

In [60]:
# Apophis
tabla_apophis, jd_apophis, pos_vel_apophis = pc.consulta_horizons(id = "99942", location = "@0", epochs = "2028-01-01")
pos_vel_apophis = pos_vel_apophis / np.array([AU, AU, AU, V_canon, V_canon, V_canon]) # Convert to canonical units

mass_apophis = 18143694800 # kg, estimated mass of Apophis
masa_apophis = mass_apophis / M_sun # Convert to canonical mass units

In [61]:
# Se crea entonces un sistema (de 2 cuerpitos mientras :p)

sistema = [
    dict(m = 1, r = [0,0,0], v = [0,0,0]), # Sun
    dict(m = masa_apophis, r = pos_vel_apophis[:3].tolist(), v = pos_vel_apophis[3:].tolist())  # Apophis
]

ts = np.linspace(0,10,100)
rs, vs, rps, vps, constantes = pc.ncuerpos_solucion(sistema, ts)


In [62]:
# graph !
fig = go.Figure()
fig.add_trace(go.Scatter3d(x=rs[0][:,0], y=rs[0][:,1], z=rs[0][:,2], mode='lines', name='Cuerpo 1'))
fig.add_trace(go.Scatter3d(x=rs[1][:,0], y=rs[1][:,1], z=rs[1][:,2], mode='lines', name='Cuerpo 2'))
fig.update_layout(title='Trayectorias de los cuerpos', scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'))
fig.show()